In [1]:
import pandas as pd

reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

negative_reviews = reviews[
    reviews["review_score"] <= 3
].copy()

negative_reviews["review_text"] = (
    negative_reviews["review_comment_message"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

negative_reviews = negative_reviews[
    negative_reviews["review_text"] != ""
].copy()

print("Negative reviews with text:", len(negative_reviews))

Negative reviews with text: 14445


In [2]:
def classify_complaint(text):
    text = text.lower()

    if any(word in text for word in [
        "atras", "atrasado", "demora", "demorou", "demorando",
        "prazo", "tarde", "não chegou", "nao chegou",
        "ainda não recebi", "ainda nao recebi"
    ]):
        return "Late delivery"

    elif any(word in text for word in [
        "danificado", "danificada", "quebrado", "quebrada",
        "amassado", "amassada", "avariado", "defeito"
    ]):
        return "Product damage"

    elif any(word in text for word in [
        "produto errado", "produto diferente", "mandaram outro",
        "veio outro", "recebi outro", "errado"
    ]):
        return "Wrong product"

    elif any(word in text for word in [
        "faltando", "faltou", "falta", "incompleto",
        "não recebi", "nao recebi", "não veio", "nao veio"
    ]):
        return "Missing product"

    elif any(word in text for word in [
        "qualidade", "péssima", "pessima", "ruim",
        "inferior", "não funciona", "nao funciona",
        "defeito", "fraco", "fraca"
    ]):
        return "Product quality"

    elif any(word in text for word in [
        "vendedor", "vendedora", "loja", "atendimento",
        "responde", "resposta", "suporte"
    ]):
        return "Seller issue"

    elif any(word in text for word in [
        "embalagem", "embalado", "pacote", "caixa",
        "embalagem ruim", "mal embalado"
    ]):
        return "Packaging"

    elif any(word in text for word in [
        "frete", "transportadora", "envio", "shipping",
        "rastreio", "rastreamento"
    ]):
        return "Shipping"

    elif any(word in text for word in [
        "pagamento", "paguei", "cobrança", "cobranca",
        "cartão", "cartao", "reembolso", "estorno"
    ]):
        return "Payment"

    else:
        return "Other"


negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(classify_complaint)
)

print(negative_reviews["complaint_category"].value_counts())

complaint_category
Other              5634
Late delivery      3101
Missing product    2115
Seller issue        911
Product quality     830
Product damage      630
Wrong product       403
Packaging           286
Shipping            279
Payment             256
Name: count, dtype: int64


In [3]:
other_reviews = negative_reviews[
    negative_reviews["complaint_category"] == "Other"
]

print(
    other_reviews[
        ["review_score", "review_text"]
    ].sample(20, random_state=42).to_string(index=False)
)

 review_score                                                                                                                                                                                       review_text
            2                                                                                                                         não gostei. muito fina. não é cara, mas por esse preço compro uma melhor.
            3                                                                                                        bonito, mas o som da voz não é muito bom, tem duas músicas apenas é uma q nao é de criança
            2                                                                                                                                                                                      gostei muito
            3                                                        o produto não é igual ao mostrado na imagem anunciada. estou esperando o tablet chegar, caso não ca

In [4]:
def classify_complaint(text):
    text = text.lower()

    # 1. Missing product / incomplete order
    if any(word in text for word in [
        "faltando", "faltou", "falta",
        "recebi só", "recebi apenas", "recebido apenas",
        "apenas 1", "só 1", "só uma",
        "não recebi", "nao recebi",
        "não veio", "nao veio",
        "pedido incompleto", "entrega incompleta",
        "não foi recebido", "nao foi recebido",
        "não foi entregue", "nao foi entregue",
        "consta como entregue"
    ]):
        return "Missing product"

    # 2. Wrong product
    elif any(word in text for word in [
        "produto errado", "produto diferente",
        "produto não é igual", "produto nao é igual",
        "mandaram outro", "mandaram uma outra",
        "veio outro", "recebi outro",
        "veio diferente", "recebi diferente",
        "não corresponde", "nao corresponde",
        "modelo errado", "tamanho errado",
        "cor errada", "veio errado"
    ]):
        return "Wrong product"

    # 3. Product damage
    elif any(word in text for word in [
        "danificado", "danificada",
        "quebrado", "quebrada",
        "amassado", "amassada",
        "avariado", "avariada",
        "estragou", "quebrou",
        "chegou quebrado"
    ]):
        return "Product damage"

    # 4. Late delivery
    elif any(word in text for word in [
        "atras", "atrasado", "atrasada",
        "demora", "demorou", "demorando",
        "demorado", "demorada",
        "prazo", "tarde",
        "não chegou", "nao chegou",
        "ainda não recebi", "ainda nao recebi",
        "espera", "esperando",
        "não foi entregue", "nao foi entregue",
        "mais de 10 dias", "mais de um mês",
        "pontualidade"
    ]):
        return "Late delivery"

    # 5. Product quality
    elif any(word in text for word in [
        "qualidade", "péssima", "pessima",
        "ruim", "inferior",
        "não funciona", "nao funciona",
        "defeito", "defeituoso",
        "fraco", "fraca",
        "fina", "fino",
        "pequeno", "pequena",
        "tamanho", "não gostei", "nao gostei",
        "não serviu", "nao serviu",
        "não encaixa", "nao encaixa",
        "não é perfeito", "nao é perfeito"
    ]):
        return "Product quality"

    # 6. Seller issue
    elif any(word in text for word in [
        "vendedor", "vendedora",
        "loja", "atendimento",
        "responde", "resposta",
        "suporte", "central",
        "não consigo contato", "nao consigo contato",
        "não respond", "nao respond"
    ]):
        return "Seller issue"

    # 7. Packaging
    elif any(word in text for word in [
        "embalagem", "embalado",
        "pacote", "caixa",
        "mal embalado", "embalagem ruim"
    ]):
        return "Packaging"

    # 8. Shipping / tracking
    elif any(word in text for word in [
        "frete", "transportadora",
        "envio", "rastreio",
        "rastreamento", "transporte"
    ]):
        return "Shipping"

    # 9. Payment / refund
    elif any(word in text for word in [
        "pagamento", "paguei",
        "cobrança", "cobranca",
        "cartão", "cartao",
        "reembolso", "estorno"
    ]):
        return "Payment"

    else:
        return "Other"


negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(classify_complaint)
)

print(negative_reviews["complaint_category"].value_counts())

complaint_category
Other              4660
Missing product    3722
Late delivery      2430
Product quality    1377
Seller issue        757
Wrong product       439
Product damage      378
Shipping            245
Packaging           228
Payment             209
Name: count, dtype: int64


In [5]:
other_reviews = negative_reviews[
    negative_reviews["complaint_category"] == "Other"
]

print(
    other_reviews[
        ["review_score", "review_text"]
    ].sample(30, random_state=42).to_string(index=False)
)

 review_score                                                                                                                                                                                             review_text
            3                                                                                                                                                                                  pelo preço vale a pena
            1 o produto que consta no pedido e na nota fiscal não é o que foi entregue a mim. o pedido e nf são para capa case carteira preta, porém foi entregue película + capa em gel transparente. quero a troca.
            1                                                                                                                                                                                        impossível usalo
            1                                                                                                                                   

In [6]:
def classify_complaint(text):
    text = text.lower()

    # 1. Missing product
    if any(word in text for word in [
        "faltando", "faltou", "falta",
        "recebi só", "recebi apenas", "recebido apenas",
        "apenas 1", "só 1", "só uma",
        "não recebi", "nao recebi",
        "não veio", "nao veio",
        "pedido incompleto", "entrega incompleta",
        "somente recebi", "apenas uma",
        "não foram entregue", "nao foram entregue"
    ]):
        return "Missing product"

    # 2. Wrong product
    elif any(word in text for word in [
        "produto errado", "produto diferente",
        "não é o que foi", "nao e o que foi",
        "não é dos", "não foi o mesmo",
        "nao foi o mesmo",
        "não é igual", "nao e igual",
        "mandaram outro", "veio outro",
        "recebi outro", "veio diferente",
        "recebi diferente", "cor errada",
        "recebi um preto", "recebi um branco",
        "modelo errado", "tamanho errado",
        "cor diferente", "capa ficou maior"
    ]):
        return "Wrong product"

    # 3. Product damage
    elif any(word in text for word in [
        "danificado", "danificada",
        "quebrado", "quebrada",
        "amassado", "amassada",
        "avariado", "avariada",
        "estragou", "quebrou",
        "solda", "se desfez",
        "riscado", "riscada"
    ]):
        return "Product damage"

    # 4. Late delivery
    elif any(word in text for word in [
        "atras", "atrasado", "atrasada",
        "demora", "demorou", "demorando",
        "demorado", "demorada",
        "prazo", "tarde",
        "não chegou", "nao chegou",
        "mercadoria não chegou", "mercadoria nao chegou",
        "ainda não recebi", "ainda nao recebi",
        "ainda estou aguardando",
        "aguardando", "esperando",
        "entrega ainda", "não foi entregue",
        "nao foi entregue",
        "pontualidade"
    ]):
        return "Late delivery"

    # 5. Cancellation / refund
    elif any(word in text for word in [
        "cancelar minha compra",
        "cancelar a compra",
        "compra cancelada",
        "foi cancelado",
        "cancelamento",
        "dinheiro reembolsado",
        "reembolso",
        "devolver o dinheiro",
        "devolução do dinheiro",
        "devolver o produto"
    ]):
        return "Cancellation / Refund"

    # 6. Product quality
    elif any(word in text for word in [
        "qualidade", "péssima", "pessima",
        "ruim", "inferior",
        "não funciona", "nao funciona",
        "defeito", "defeituoso",
        "fraco", "fraca",
        "fina", "fino",
        "pequeno", "pequena",
        "tamanho", "medidas",
        "não gostei", "nao gostei",
        "não serviu", "nao serviu",
        "não encaixa", "nao encaixa",
        "acabamento", "ferrugem",
        "usado", "utilizado",
        "sujo", "bolhas",
        "mal colado",
        "não é dos melhores"
    ]):
        return "Product quality"

    # 7. Seller issue
    elif any(word in text for word in [
        "vendedor", "vendedora",
        "loja", "atendimento",
        "responde", "resposta",
        "suporte", "central",
        "não consigo contato", "nao consigo contato",
        "não respond", "nao respond"
    ]):
        return "Seller issue"

    # 8. Packaging
    elif any(word in text for word in [
        "embalagem", "embalado",
        "pacote", "caixa",
        "mal embalado", "embalagem ruim"
    ]):
        return "Packaging"

    # 9. Shipping / tracking
    elif any(word in text for word in [
        "frete", "transportadora",
        "envio", "rastreio",
        "rastreamento", "transporte",
        "status de onde", "acompanhar o produto"
    ]):
        return "Shipping"

    # 10. Payment
    elif any(word in text for word in [
        "pagamento", "paguei",
        "cobrança", "cobranca",
        "cartão", "cartao",
        "estorno", "parcelas pagas"
    ]):
        return "Payment"

    else:
        return "Other"


negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(classify_complaint)
)

print(negative_reviews["complaint_category"].value_counts())

complaint_category
Other                    4391
Missing product          3313
Late delivery            3004
Product quality          1496
Seller issue              689
Product damage            409
Wrong product             350
Shipping                  236
Cancellation / Refund     200
Packaging                 198
Payment                   159
Name: count, dtype: int64


In [7]:
other_reviews = negative_reviews[
    negative_reviews["complaint_category"] == "Other"
]

print(
    other_reviews[
        ["review_score", "review_text"]
    ].sample(20, random_state=100)
    .to_string(index=False)
)

 review_score                                                                                                                                                                         review_text
            1                                                                                                              acho que se pago frente tenho que recebero produto na minha residência
            1                                                                                                                             somente 1 produto meu chegou gostaria de saber do outro
            1                                                                   comprei dois fones de de ouvido e só recebi um, ainda sem funcionar, estarei devolvendo na próxima segunda feira.
            3                                                                                                                                                     relógio não é o que eu esperava
            2                 

In [8]:
def classify_complaint(text):
    text = text.lower()

    # 1. Missing product
    if any(word in text for word in [
        "faltando", "faltou", "falta",
        "recebi só", "recebi apenas", "recebido apenas",
        "só recebi", "so recebi",
        "apenas 1", "só 1", "so 1", "só uma",
        "somente 1", "somente um",
        "não recebi", "nao recebi",
        "não veio", "nao veio",
        "pedido incompleto", "entrega incompleta",
        "produto incompleto", "veio incompleto",
        "somente recebi",
        "não foram entregue", "nao foram entregue",
        "sem o bebê conforto",
        "do outro"
    ]):
        return "Missing product"

    # 2. Wrong product
    elif any(word in text for word in [
        "produto errado", "produto diferente",
        "não é o que foi", "nao e o que foi",
        "não é igual", "nao e igual",
        "não foi o mesmo", "nao foi o mesmo",
        "mandaram outro", "veio outro",
        "recebi outro", "veio diferente",
        "recebi diferente",
        "livro errado",
        "cor errada", "cor diferente",
        "modelo errado", "tamanho errado",
        "veio com varias cores",
        "veio com várias cores"
    ]):
        return "Wrong product"

    # 3. Product damage
    elif any(word in text for word in [
        "danificado", "danificada",
        "quebrado", "quebrada",
        "amassado", "amassada",
        "avariado", "avariada",
        "estragou", "quebrou",
        "solda", "se desfez",
        "riscado", "riscada"
    ]):
        return "Product damage"

    # 4. Late delivery
    elif any(word in text for word in [
        "atras", "atrasado", "atrasada",
        "demora", "demorou", "demorando",
        "demorado", "demorada",
        "prazo", "tarde",
        "não chegou", "nao chegou",
        "mercadoria não chegou",
        "mercadoria nao chegou",
        "ainda não recebi", "ainda nao recebi",
        "ainda estou aguardando",
        "estou no aguardo",
        "aguardo", "aguardando",
        "esperando",
        "chegou depois",
        "não foi entregue", "nao foi entregue",
        "pontualidade"
    ]):
        return "Late delivery"

    # 5. Cancellation / Refund
    elif any(word in text for word in [
        "cancelar minha compra",
        "cancelar a compra",
        "compra cancelada",
        "foi cancelado",
        "cancelamento",
        "dinheiro reembolsado",
        "reembolso",
        "devolver o dinheiro",
        "devolução do dinheiro",
        "devolver o produto"
    ]):
        return "Cancellation / Refund"

    # 6. Product quality
    elif any(word in text for word in [
        "qualidade", "péssima", "pessima",
        "ruim", "inferior",
        "não funciona", "nao funciona",
        "sem funcionar",
        "defeito", "defeituoso",
        "fraco", "fraca",
        "fina", "fino",
        "pequeno", "pequena",
        "tamanho", "medidas",
        "não gostei", "nao gostei",
        "não serviu", "nao serviu",
        "não encaixa", "nao encaixa",
        "acabamento", "ferrugem",
        "usado", "utilizado",
        "sujo", "bolhas",
        "mal colado",
        "não é dos melhores",
        "não protege", "nao protege",
        "não mantém", "nao mantém",
        "não mantem", "nao mantem",
        "falso", "falsa",
        "não era o que esperava",
        "nao era o que esperava"
    ]):
        return "Product quality"

    # 7. Seller issue
    elif any(word in text for word in [
        "vendedor", "vendedora",
        "loja", "atendimento",
        "responde", "resposta",
        "suporte", "central",
        "contactar", "contato",
        "não consigo contato",
        "nao consigo contato",
        "não obtive sucesso",
        "nao obtive sucesso",
        "não respond", "nao respond"
    ]):
        return "Seller issue"

    # 8. Packaging
    elif any(word in text for word in [
        "embalagem", "embalado",
        "pacote", "caixa",
        "mal embalado", "embalagem ruim"
    ]):
        return "Packaging"

    # 9. Shipping / tracking
    elif any(word in text for word in [
        "frete", "transportadora",
        "envio", "rastreio",
        "rastreamento", "transporte",
        "status de onde",
        "acompanhar o produto",
        "entrega na residência",
        "correios"
    ]):
        return "Shipping"

    # 10. Payment
    elif any(word in text for word in [
        "pagamento", "paguei",
        "cobrança", "cobranca",
        "cartão", "cartao",
        "estorno", "parcelas pagas",
        "banco"
    ]):
        return "Payment"

    else:
        return "Other"


negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(classify_complaint)
)

print(negative_reviews["complaint_category"].value_counts())

complaint_category
Other                    3947
Missing product          3625
Late delivery            3082
Product quality          1512
Seller issue              734
Product damage            408
Wrong product             341
Shipping                  300
Cancellation / Refund     189
Packaging                 182
Payment                   125
Name: count, dtype: int64


In [9]:
complaint_summary = (
    negative_reviews["complaint_category"]
    .value_counts()
    .reset_index()
)

complaint_summary.columns = ["complaint_category", "review_count"]

complaint_summary["percentage"] = (
    complaint_summary["review_count"]
    / complaint_summary["review_count"].sum()
    * 100
).round(2)

print(complaint_summary)

       complaint_category  review_count  percentage
0                   Other          3947       27.32
1         Missing product          3625       25.10
2           Late delivery          3082       21.34
3         Product quality          1512       10.47
4            Seller issue           734        5.08
5          Product damage           408        2.82
6           Wrong product           341        2.36
7                Shipping           300        2.08
8   Cancellation / Refund           189        1.31
9               Packaging           182        1.26
10                Payment           125        0.87
